# Week 7 · Saving and history

[![Open in Colab](../assets/badges/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_07.ipynb)

A system that forgets cannot be trusted. This week your system learns to save its records safely, load them back after a restart, and keep a history that explains what happened.

| | |
|---|---|
| **This week's question** | How does a system remember, and how can it explain what happened? |
| **Python you will learn** | memory and disk, files and paths, `open` and `with`, `"w"` and `"a"`, `os.path.exists`, JSON (`json.dump` / `json.load`), JSON Lines; reading a safe save (`os.replace`) |
| **Principle 7** | **Never lose data.** Save safely, keep a history, and be able to explain what happened. |
| **File you add** | `my_foodbridge/storage.py` → `save_records` (given), `load_records`, `append_event`, `read_events` |
| **What you will see** | `bridge.show("history")`: the sample day saved with your code, the two files, a timeline of events, and a restart that loads everything back |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 7: storage.py saves safely and keeps a history](../assets/diagrams/system_map_w07.svg)

### By the end of this week you can …

- **explain** the difference between memory and disk, and what a file and its path are;
- **write and read** files with `open` and `with`, and **choose** `"w"` or `"a"` so that no data is lost;
- **save** records as JSON, **keep** a history as JSON Lines, and **explain** why a safe save writes a temporary file first;
- **decide** which events your history keeps, with a number from your own saved history as evidence.

### The story so far

Your folder holds four files: `rules.py` and `records.py` (Week 4), `data_model.py`, the check at the door (Week 5), and `lifecycle.py`, which allows only sensible moves (Week 6). Your **code** is safe: it lives in your Google Drive.

Your **data** is not. Yesterday at 4 pm Colab restarted, and the whole day was gone: every offer, approval and delivery lived only in memory. Then Sunrise Bakery phoned: *"What happened to our 30 portions of bread rolls?"* There was no file to open and no history to read. This week you add `storage.py`, so your system remembers.

## Before you start

1. **File → Save a copy in Drive.** Work in your copy.
2. Run the setup cell. In Colab it asks to connect Google Drive: say yes, so your `my_foodbridge/` folder is kept between weeks.
3. ⭐ = everyone · ⭐⭐ = stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


## Class warm-up · Research 5 exchange (10 min)

For Research 5 you read about what happens to data when the power goes off (files, databases, backups), asked an AI about what stayed unclear, checked one claim in Colab, and wrote your view on *"Should a system ever really delete data?"* Compare with two classmates:

1. Name one thing the book and the AI explained differently, or where the AI was vague or wrong. Which claim did you test in Colab, and was it right?
2. Should a system ever really delete data? Say your side and your reason in two sentences. Then listen for one good reason from the other side.
3. This week your history only grows: `append_event` adds one line and never rewrites earlier lines, and a safe save never leaves half a file. Do these two ideas fit your side from question 2? Is there anything FoodBridge should still delete one day, for example a donor's private contact?

📝 **My notes:** one idea from a classmate that I want to remember …

## Session 1 · Understand: memory, files and history (90 min)

### The idea: memory forgets, records must not

Think of a small shop. During the day the owner writes sales on a whiteboard, which is wiped clean at night. A careful owner also keeps a **sales book**: every sale is a new line in ink, with the time. A mistake is fixed with a new line ("refund, 2 pm"), never by tearing out a page.

A running program's memory is the whiteboard: your lists and dictionaries disappear when the program stops or Colab restarts. An information system needs two more things:

- **Persistence:** the records are saved in a file, so they are still there tomorrow.
- **An audit history:** a list of events that says **what happened, and when** (in a real system also **who** did it). We only ever add to it: it is **append-only**. Then, when a donor asks *"what happened to my bread rolls?"*, someone can answer.

This week's program thinking: **trace by hand** what a file holds after each step, **test the boundary** (no file yet, an empty history), and **read the error message** from its last line.

### 1 · Memory and disk: a file and its path

📖 **What it is.** **Memory** holds the values of a running program: every name, list and dictionary you make. The **disk** (a hard disk, an SSD, Google Drive) holds **files**. A file is a named piece of data on disk, often text. A **path** says where a file is: the folders it is in, then its name, joined by `/`. `my_foodbridge/demo_day/offers.json` is the file `offers.json` in the folder `demo_day`, inside your folder `my_foodbridge`. A path with no folder, such as `shopping.txt`, means "in the current folder". In Colab that is the runtime's temporary workspace (`/content`), which disappears when the runtime is deleted. Your notebook itself is saved in Drive, and so is your `my_foodbridge/` folder.

**Why it exists.** Memory is fast, but it forgets: when the program stops or the runtime restarts, memory is empty. A file on disk stays until someone deletes it, so anything you need tomorrow must be written to disk.

| | memory | disk |
|---|---|---|
| holds | the names, lists and dictionaries of a running program | files in folders |
| after a restart | empty | still there |
| in Python | `offers = [...]` | `open("offers.json", "w", encoding="utf-8")` |

![Memory and disk: a restart empties the variables in memory; the files on the disk stay](../assets/diagrams/memory_disk.svg)

**How Python runs it.** `open(path)` gives you a **file object**: a connection to the file that you read or write through. Without a mode, `open` only reads; `open(path, "w", encoding="utf-8")` opens it for **writing** (concept 2 says more), and `encoding="utf-8"` lets the file hold any language, including Khmer. `file.write(text)` puts exactly that text into the file and adds nothing: a line ends only where you write `"\n"`, the line break. `with open(...) as file:` closes the connection at the end of the block, **even when something goes wrong in the middle**. Closing matters, because it is when Python finishes writing the last text to disk. A `for` loop over a file gives you one line at a time, and each line still ends with `"\n"`.

```{iframe} https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_memory_disk.html
:width: 100%
:title: Memory forgets, files stay: json.dump, "w" and "a"

[Memory forgets, files stay: `json.dump`, `"w"` and `"a"`](https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_memory_disk.html)
```

**Where you meet it.** Your phone keeps your photos when it switches off, because they are files. A game "saves" your progress to a file. A web form that forgets what you typed after a reload kept it only in memory.

👀 **Example.** 🤔 **Predict** how many lines the loop prints, and what `line.strip()` removes.

In [ ]:
with open("shopping.txt", "w", encoding="utf-8") as file:
    file.write("rice\n")
    file.write("eggs\n")

with open("shopping.txt", encoding="utf-8") as file:
    for line in file:
        print("-", line.strip())

**Prediction:** …  **Result:** …  **Why:** …

Open the **Files** panel (the folder icon on the left): `shopping.txt` is there, a real file on disk.

### ✍️ Try it 1 · `read_lines`

Write `read_lines(path)`: return every line of the file as a list, without `"\n"`. The first lines of the cell make a second file to test with.

In [ ]:
with open("menu.txt", "w", encoding="utf-8") as file:     # a second file to test with
    file.write("fried rice\nnoodle soup\nfish amok\n")


def read_lines(path):
    lines = []
    # TODO: open path to read (encoding="utf-8"); for each line, append line.strip() to lines
    return lines


bridge.check("read_lines", read_lines, examples=[
    ("shopping.txt", ["rice", "eggs"]),
    ("menu.txt", ["fried rice", "noodle soup", "fish amok"]),
])

🐞 **Common mistake · a forgotten `"\n"`.** A classmate writes the shopping list like this:

```python
with open("shopping.txt", "w", encoding="utf-8") as file:
    file.write("rice")
    file.write("eggs")
```

No error appears, but `read_lines("shopping.txt")` now gives `['riceeggs']`: one line. `write` puts exactly the text you give it into the file, nothing more. Fix: end every line with `"\n"`. This is a wrong **output**, not an error message, so only a check with an expected answer catches it.

### 2 · `"w"` empties the file, `"a"` adds to the end

📖 **What it is.** The second argument of `open` is the **mode**: what you want to do with the file.

| mode | the file does not exist | the file exists | use it for |
|---|---|---|---|
| no mode (or `"r"`) | `FileNotFoundError` | read it from the start | reading |
| `"w"` (write) | creates it | **empties it** the moment it opens | a file you replace as a whole: the records now |
| `"a"` (append) | creates it | keeps everything, adds to the end | a file that only grows: a history |

**Why it exists.** Programs do two different jobs with files. Sometimes the whole content is replaced by a newer version: today's list of offers. Sometimes one new line is added after everything before it: today's events. The mode tells Python which job you mean.

**How Python runs it.** With `"w"`, the file is emptied **when `open` runs**, before you write anything, and Python gives no warning. With `"a"`, every `write` goes to the end, and earlier lines are never touched. Opening a file in the wrong mode is one of the most common ways programs lose data.

![File modes: &quot;w&quot; empties the file first, then writes; &quot;a&quot; keeps every line and adds to the end](../assets/diagrams/write_append.svg)

**Where you meet it.** A bank statement only grows: each payment is a new line (`"a"`). Your phone's settings file is written again as a whole each time you change a setting (`"w"`).

👀 **Example.** 🤔 **Predict** what the cell prints. The file starts with *maths* and *English*. (`file.read()` gives the whole file as one text.)

In [ ]:
with open("homework.txt", "w", encoding="utf-8") as file:
    file.write("maths\nEnglish\n")
with open("homework.txt", "w", encoding="utf-8") as file:     # "w" again
    file.write("history\n")
with open("homework.txt", "a", encoding="utf-8") as file:     # "a"
    file.write("Khmer\n")
with open("homework.txt", encoding="utf-8") as file:
    print(file.read())

**Prediction:** …  **Result:** …  **Why:** …

### ✍️ Try it 2 · `add_sale`

The shop's sales book starts the day with one sale. Write `add_sale(path, sale)`: add `sale` as a new line at the **end** of the file, then return every line of the file. Your `read_lines` already does the second part, so call it: **decompose** a job into parts you already have.

One example is given. The examples run in order, and each one adds a sale. So **trace by hand** what the whole book holds after your next sale, and write that as the expected answer.

In [ ]:
with open("sales_book.txt", "w", encoding="utf-8") as file:   # the day starts with one sale
    file.write("08:00 two bread rolls\n")


def add_sale(path, sale):
    # TODO 1: open path with "a" (and encoding="utf-8") and write sale + "\n"
    # TODO 2: return read_lines(path)
    return []


bridge.check("add_sale", add_sale, examples=[
    (("sales_book.txt", "09:15 one rice box"), ["08:00 two bread rolls", "09:15 one rice box"]),
    # ✍️ add your own: the next sale (trace the whole book by hand) and an odd case, an empty sale ""
    # (("sales_book.txt", "..."), ["08:00 two bread rolls", "09:15 one rice box", "..."]),
    # (("sales_book.txt", ""), [...]),
])

🐞 **Common mistake · `"w"` wiped the morning's sales.** A classmate opens the sales book with `"w"` in `add_sale`. There is no error message. The wrong output is the only sign:

```text
❌ add_sale('sales_book.txt', '09:15 one rice box') → ['09:15 one rice box']   (expected ['08:00 two bread rolls', '09:15 one rice box'])
```

It means: `"w"` emptied the book the moment it opened, so the 08:00 sale is gone, and Python never warned. Fix: use `"a"` for any file that only grows. A test catches it when it adds **twice** and checks that the **first** line is still there.

### 3 · JSON: records as text, and "is the file there?"

📖 **What it is.** A **text file** holds only text, not Python lists and dictionaries. Your records are lists and dictionaries, so they must become text first. **JSON** is a standard way to write them as text. It looks almost like Python:

| Python | JSON text |
|---|---|
| `{"id": "F001", "portions": 30}` | `{"id": "F001", "portions": 30}` |
| `True`, `False`, `None` | `true`, `false`, `null` |
| a list of dictionaries `[{...}, {...}]` | a list of objects `[{...}, {...}]` |

**Why it exists.** In Week 5 every value from the CSV file arrived as text, so you needed `int()`. JSON keeps numbers as numbers, and it can hold lists inside dictionaries. People can read it, and almost every programming language can load it. (This notebook is JSON too: an `.ipynb` file is JSON text.)

**How Python runs it.** `json.dump(data, file)` turns your lists and dictionaries into JSON text and writes it into the open file. `json.load(file)` reads the text and builds **new** lists and dictionaries in memory. `indent=2` puts each item on its own line, so a person can read the file. With an `s` at the end, `json.dumps(data)` and `json.loads(text)` do the same with a **string** instead of a file. You need them when you write one line at a time.

![JSON: json.dump writes a list of dictionaries as text in offers.json; json.load reads it back](../assets/diagrams/json_text.svg)

Before you read a file, ask whether it is there: `os.path.exists(path)` answers `True` or `False`. On the first day of a new system there is no file yet, and that is normal, not an error. The pattern you will use all week: **if the file exists, read it; if not, start with an empty list.**

**Where you meet it.** When a weather app asks a server for today's forecast, the answer comes back as JSON text.

👀 **Example.** 🤔 **Predict** the two printed values: is `loaded[1]["students"] + 1` a number, or an error (text + number)? Does `diary.json` exist?

In [ ]:
import json
import os

classes = [{"code": "MIS101", "students": 42}, {"code": "ACC102", "students": 35}]
with open("classes.json", "w", encoding="utf-8") as file:
    json.dump(classes, file, indent=2)
with open("classes.json", encoding="utf-8") as file:
    loaded = json.load(file)
print(loaded[1]["students"] + 1, os.path.exists("diary.json"))

**Prediction:** …  **Result:** …  **Why:** …

Open `classes.json` in the **Files** panel (double-click it): it is plain text that you can read.

### ✍️ Try it 3 · `load_list`

Write `load_list(path)`: when there is no file at `path`, return `[]`; otherwise return the list saved in the file. This is how a new system starts on its first day, and it is almost your `load_records`. **Test the boundary:** the first day, when nothing is saved yet.

In [ ]:
def load_list(path):
    # TODO 1: if there is no file at path, return []   (use os.path.exists)
    # TODO 2: otherwise open the file and return json.load(file)
    return None


bridge.check("load_list", load_list, examples=[
    ("classes.json", [{"code": "MIS101", "students": 42}, {"code": "ACC102", "students": 35}]),
    # ✍️ add your own: one boundary case (nothing saved yet) and one odd case (a folder that does not exist)
    # ("....json", ...),
    # ("no_such_folder/....json", ...),
])

🐞 **Common mistake · `FileNotFoundError`.** Without the `os.path.exists` line, the first day stops the program:

```text
FileNotFoundError: [Errno 2] No such file or directory: 'nothing_saved.json'
```

**Read the error message from its last line:** first the kind of error, then what went wrong, then the path Python looked for. It means there is no file at that path. On the first day that is normal, so check with `os.path.exists` first and return `[]`. If the file *should* be there, compare the path with the **Files** panel letter by letter: a missing folder or a typo (`offer.json` for `offers.json`) gives the same error.

### 4 · JSON Lines: a history you only add to

📖 **What it is.** A history is a list of **events**, and it has a different shape from the records. Each event is one small JSON object on **its own line**. This is called **JSON Lines**, and the files end in `.jsonl`.

**Why it exists.** The records say what is true **now**, so each save replaces the whole file. The history says **what happened**, so it only grows: one new line per event, and earlier lines are never rewritten. A mistake is fixed with a new event, like a new line in the sales book.

| | `offers.json` (the records) | `history.jsonl` (the history) |
|---|---|---|
| says | what is true now | what happened, in order |
| a save | replaces the whole file (`"w"`) | adds one line at the end (`"a"`) |
| inside | one JSON list | one JSON object per line |
| in Python | `json.dump` / `json.load` | `json.dumps(event) + "\n"` / `json.loads(line)` |

**How Python runs it.** To add an event, open the file with `"a"` and write `json.dumps(event) + "\n"`: one dictionary becomes one line of text. To read the history, loop over the file and turn each line back into a dictionary with `json.loads(line)`. Adding a line never touches the lines before it, so a crash can damage at most the last line, never the whole history.

![JSON Lines: one event per line in history.jsonl; append_event adds a line, read_events reads them all](../assets/diagrams/json_lines.svg)

**Where you meet it.** A bank statement, the message history of a chat app and the log of a web server are all append-only histories.

👀 **Example.** 🤔 **Predict, then try:** run the cell **twice**. How many lines does the history have then? Is that right for a history? Would it be right for a list of records?

In [ ]:
def log_event(path, event):
    with open(path, "a", encoding="utf-8") as file:     # "a": add to the end, never empty
        file.write(json.dumps(event) + "\n")             # one event = one line


log_event("phone_history.jsonl", {"time": "09:00", "event": "top up", "usd": 5})
with open("phone_history.jsonl", encoding="utf-8") as file:
    for line in file:
        print(json.loads(line))

**Prediction:** …  **Result:** …  **Why:** …

### ✍️ Try it 4 · `log_sale` (JSON Lines)

The shop now keeps its sales log as JSON Lines: one sale per line, each a small dictionary with the price in riel. Write `log_sale(path, sale)`: add `sale` to the end of the log as one JSON line, then read the whole log back and return the day's total in riel. The first lines of the cell start the day with one sale.

As in Try it 2, the examples run in order and each one adds a sale. So **trace by hand** the day's total after your next sale.

In [ ]:
with open("sales_log.jsonl", "w", encoding="utf-8") as file:     # the day starts with one sale
    file.write(json.dumps({"time": "08:00", "item": "bread rolls", "riel": 4000}) + "\n")


def log_sale(path, sale):
    # TODO 1: open path with "a" (encoding="utf-8") and write json.dumps(sale) + "\n"
    total = 0
    # TODO 2: open path to read; for each line, json.loads(line) gives one sale: add its "riel" to total
    return total


bridge.check("log_sale", log_sale, examples=[
    (("sales_log.jsonl", {"time": "09:15", "item": "rice box", "riel": 6000}), 10000),
    # ✍️ add your own: the next sale (trace the day's total by hand) and an odd case, a free sale of 0 riel
    # (("sales_log.jsonl", {"time": "...", "item": "...", "riel": ...}), ...),
    # (("sales_log.jsonl", {"time": "...", "item": "...", "riel": 0}), ...),
])

🐞 **Common mistake · `JSONDecodeError` from a half-written line.** The power goes off while a line is being written, so the last line of a history is only `{"step": 3, "event": "add`. Reading it back stops with:

```text
json.decoder.JSONDecodeError: Unterminated string starting at: line 1 column 22 (char 21)
```

A blank line gives a similar message: `json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)`. It means the text is not complete JSON, so `json.loads` cannot build a dictionary from it. Fix: skip blank lines when you read (`if line.strip() != "":`), as your `read_events` will. A half-written line is also the reason for Session 2's **safe save**: in a JSON Lines history a power cut can damage only the last line, but a records file written straight with `"w"` can be broken as a whole.

### ✍️ Try it 5 · `count_events` (FoodBridge)

In FoodBridge each event in the history is a dictionary. Its `"event"` key says what kind of event it was: `"added"`, `"input_rejected"` or `"moved"`. Write `count_events(events)`: return a dictionary `{kind of event: how many}`. It is the counting pattern from Week 6: `counts.get(kind, 0) + 1`. You will meet the same counts again in Session 3, as the evidence for your decision.

In [ ]:
def count_events(events):
    counts = {}
    # TODO: for each event, add 1 to counts[event["event"]]   (counts.get(kind, 0) + 1)
    return counts


short_day = [
    {"step": 1, "event": "added", "offer": "F001"},
    {"step": 2, "event": "added", "offer": "F002"},
    {"step": 3, "event": "input_rejected", "offer": ""},
    {"step": 4, "event": "moved", "offer": "F002"},
]
bridge.check("count_events", count_events, examples=[
    (short_day, {"added": 2, "input_rejected": 1, "moved": 1}),
    # ✍️ add your own: one boundary case (an empty history) and one odd case (a history with only one kind)
    # ([], ...),
    # ([{"step": 1, "event": "...", "offer": "F004"}], ...),
])

### 🔁 Explain it back

Answer in your own words, without code. Two or three sentences each.

1. What is the difference between memory and disk? After a restart, what is left?
2. When do you open a file with `"w"`, and when with `"a"`? Name one FoodBridge file for each.
3. Why is the history a JSON Lines file that only grows, and not one JSON list saved again each time?

📝 **My answers:** 1 … · 2 … · 3 …

## Session 2 · Practise, then look under the hood (90 min)

*New runtime since Session 1? Run the setup cell at the top first, so `bridge` is ready.*

### Part A · Practise (≈ 45 min)

#### 🚨 Incident drill: the runtime is gone (15 min)

A real team practises a disaster **before** it happens. First check that the setup cell said your folder is in **Google Drive**. Without Drive, skip step 2: deleting the runtime would delete your `my_foodbridge/` files too.

1. 🤔 **Predict:** which of these survive a deleted runtime? `shopping.txt`, `classes.json`, your `my_foodbridge/rules.py`, the variable `short_day`.
2. **Runtime → Disconnect and delete runtime.** Reconnect, run **only** the setup cell, then the cell below.
3. Compare with your prediction. What was the backup? What would Sunrise Bakery lose if FoodBridge kept its offers only in memory?

In [ ]:
import os

print("shopping.txt, a practice file:", os.path.exists("shopping.txt"))
print("classes.json, a practice file:", os.path.exists("classes.json"))
print("my_foodbridge/rules.py, in Drive:", os.path.exists("my_foodbridge/rules.py"))
# print(short_day)      # remove the # to ask memory for short_day; read the error, then put the # back

📝 **My notes:** what survived, what was lost, and why.

#### 🎫 This week's ticket

> **From:** the coordinator · **To:** you
> "Yesterday Colab restarted at 4 pm and the whole day was gone. Sunrise Bakery phoned about its 30 bread rolls, and I could not say what had happened. I need the system to save every change, load it all back in the morning, and keep a history I can read."

| | |
|---|---|
| **File** | `my_foodbridge/storage.py` |
| **Setting** | `LOG_EVENTS`: the kinds of event the history keeps (default `["added", "input_rejected", "moved"]`) |
| **How the system uses it** | after every change it calls your `save_records`; for every event of a kind in `LOG_EVENTS` it calls your `append_event`; when it starts, it calls your `load_records` and `read_events` |
| **Done when** | the four checks pass, and `bridge.show("history")` shows your parts ✅ and a restart test that gets the same 4 records back |

| Call | What must happen |
|---|---|
| `save_records(records, "offers.json")` | the file holds exactly that list; saving again **replaces** it (given to you, complete) |
| `load_records("offers.json")` | the same list back, numbers still numbers |
| `load_records("nothing_saved.json")` | ✍️ ? |
| `append_event("history.jsonl", event)`, called twice | ✍️ ? (how many lines, and which line is first?) |
| `read_events("history.jsonl")` | every event, oldest first |
| `read_events("nothing_logged.jsonl")` | `[]` |

*Watch the order of the inputs: `save_records(records, path)` takes the data first, while `append_event(path, event)` takes the path first.*

#### ✏️ Model it on paper first

**Write the expected answer first.** Fill in the two ✍️ rows of the ticket in the cell below. Then draw two boxes on paper, `offers.json` and `history.jsonl`, and let three things happen:

1. F001, 30 portions of bread rolls, is added.
2. A row is refused at the door: "portions must be a whole number".
3. F001 is approved for Hope Children's Centre.

After each step, draw what is inside each file. Which file is **replaced** each time? Which one only **grows**?

📝 **My expected answers:** `load_records("nothing_saved.json")` → … · `append_event("history.jsonl", event)`, called twice → …

📝 **My drawing, step by step** (or paste a photo here): …

<details><summary>Show the expected answers</summary>

- `load_records("nothing_saved.json")` → `[]`: no file yet is normal on the first day.
- `append_event(...)` called twice → two lines: the first event is still line 1, the second is line 2.
- `offers.json`: step 1 `[F001 new, 30]` · step 2 the same (a refused row is not a record) · step 3 `[F001 approved]`. It is **replaced** as a whole each time.
- `history.jsonl`: step 1 one line (added F001) · step 2 two lines (+ input_rejected) · step 3 three lines (+ moved F001). It only **grows**.

</details>

#### 🧩 Put the lines in order

The volunteer team keeps a sign-in history as JSON Lines. Click the lines into order. One line does not belong.

In [ ]:
bridge.parsons("sign_in_history")

<details><summary>Show the puzzle answer (if reading on the website)</summary>

```python
def read_sign_ins(path):
    sign_ins = []
    if not os.path.exists(path):
        return sign_ins
    with open(path, encoding="utf-8") as file:
        for line in file:
            sign_ins.append(json.loads(line))
    return sign_ins
```

The distractor line `with open(path, "w", encoding="utf-8") as file:` does not belong: `"w"` empties the file the moment it opens, which would erase every sign-in.

</details>

#### 🧩 From a blank cell: `refused_rows`

The coordinator wants to see why rows were refused at the door today. Write `refused_rows(path)`: read the FoodBridge history file at `path` and return the `"detail"` of every `"input_rejected"` event, oldest first. Return `[]` when there is no file.

In [ ]:
import json
import os

practice_day = [
    {"step": 1, "event": "added", "offer": "F001", "detail": "30 portions of Bread rolls"},
    {"step": 2, "event": "input_rejected", "offer": "", "detail": "portions must be a whole number, for example 12."},
    {"step": 3, "event": "input_rejected", "offer": "", "detail": "donor is required."},
    {"step": 4, "event": "moved", "offer": "F001", "detail": "new -> approved for Hope Children's Centre"},
]
with open("practice_history.jsonl", "w", encoding="utf-8") as file:   # a short history to test with
    for event in practice_day:
        file.write(json.dumps(event) + "\n")


def refused_rows(path):
    # Your code here.
    return None


bridge.check("refused_rows", refused_rows, examples=[
    ("practice_history.jsonl", ["portions must be a whole number, for example 12.", "donor is required."]),
    ("no_history.jsonl", []),
])

### Part B · Under the hood (≈ 45 min)

#### 🧑‍🏫 How `storage.py` works

Your file has four functions, and the **system** calls them. This is the data flow on the sample day:

```text
 the system (memory)                              my_foodbridge/demo_day/ (disk)
 an offer is added or moved ── save_records ──▶  offers.json.tmp ── os.replace ──▶ offers.json   (replaced)
 an event of a kind in LOG_EVENTS ── append_event ───────────────────────────────▶ history.jsonl (+1 line)

 after a restart:   load_records ◀── offers.json        read_events ◀── history.jsonl
```

**The safe save.** What happens if the power goes off while `offers.json` is being written? With `open("offers.json", "w")`, the file is emptied first. For a moment the **only** copy of every record is half-written. If the power cuts then, the next morning `json.load` meets broken text (the `JSONDecodeError` from Session 1) and the system cannot start. Every offer is lost, not only the last change.

The safe save never writes into the real file:

1. Write the complete new list into a **temporary file**, `offers.json.tmp`, in the same folder.
2. Swap it in with `os.replace("offers.json.tmp", "offers.json")`.

`os.replace` makes the name `offers.json` point to the new file **in one step**. There is no moment when `offers.json` is half old and half new. So a power cut can hit only the temporary file. If it stops during step 1, `offers.json` is still the last complete file: at worst the last change is lost, never everything. The temporary file stays in the same folder because this one-step swap works only on one disk.

```{iframe} https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_safe_save.html
:width: 100%
:title: The safe save: a temporary file, then os.replace

[The safe save: a temporary file, then `os.replace`](https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_safe_save.html)
```

![Safe save: write a temporary file, then replace the old file in one step](../assets/diagrams/safe_save.svg)

#### The Session 3 starter, line by line

In Session 3, `save_records` is given to you complete. Read it line by line:

```python
def save_records(records, path):
    """Save every record as JSON. Write a temporary file first, then swap it in."""
    temp_path = path + ".tmp"                                # offers.json → offers.json.tmp
    with open(temp_path, "w", encoding="utf-8") as file:     # "w" is safe here: it empties only the temporary file
        json.dump(records, file, indent=2)
    os.replace(temp_path, path)                              # swap it in, in one step
```

You write the other three. Their contracts, and why they are so:

| Function | Receives | Returns | Why the contract is so |
|---|---|---|---|
| `save_records(records, path)` | the list of records, and where to save it | nothing (`None`): the file is the result | the whole list is saved each time, because the records say what is true **now** |
| `load_records(path)` | where to look | the saved list, or `[]` when there is no file | `[]`, not `None`: on the first day the system simply starts with no offers, and a loop over `[]` does nothing |
| `append_event(path, event)` | where the history is, and one event (a dictionary) | nothing: the file has one more line | `"a"`, so earlier lines are never rewritten. It writes **every** event it is given: the system reads `LOG_EVENTS` and decides which events to send |
| `read_events(path)` | where the history is | every event, oldest first, or `[]` when there is no file | it skips blank lines, so one empty line cannot stop the system |

🤔 **What happens if …** Predict, then open the answer below.

1. … `append_event` opened the history with `"w"` instead of `"a"`? What does the history hold after the sample day's 16 events?
2. … the power goes off just **after** `json.dump` and just **before** `os.replace`? Which files are in `demo_day/`, and what does `load_records` return the next morning?

📝 **Prediction 1:** …  **Prediction 2:** …

<details><summary>Show what happens</summary>

1. Only the last event, #16 (F001 approved -> picked_up). Every `"w"` emptied the file before writing, so 15 events are gone, and nobody can say why F003 was rejected. The kit's check catches it: after two calls it expects two lines, and finds one.
2. `offers.json` (the last complete file) and `offers.json.tmp` (the new file, complete but never swapped in). `load_records` reads `offers.json`, so the system starts with the records of the last finished save: the last change is lost, but nothing is broken. The next save writes the temporary file again and swaps it in.

</details>

🔭 **In real work.** Databases save the same way, with **transactions**: a change is saved all or nothing, never half, the same idea as your temporary file and `os.replace`. Real systems also keep an **audit log** of who changed what and when, and **backups** in another place, so one broken disk cannot lose everything.

#### ⚖️ Good or risky?

You asked an AI assistant: *"Write a function that saves the offers and today's events at the end of the day. Keep the files small."* It gave you **Version A**. **Version B** uses this week's pieces. On a good day, both run without errors.

**Version A** (from the AI assistant)

```python
def save_day(records, todays_events):
    with open("offers.json", "w") as file:
        json.dump(records, file)
    # keep the history small: only today's events
    with open("history.jsonl", "w") as file:
        for event in todays_events:
            file.write(json.dumps(event) + "\n")
```

**Version B**

```python
def save_day(records, todays_events):
    temp_path = "offers.json.tmp"
    with open(temp_path, "w", encoding="utf-8") as file:   # 1. write the new data here
        json.dump(records, file, indent=2)
    os.replace(temp_path, "offers.json")                    # 2. swap it in, in one step
    with open("history.jsonl", "a", encoding="utf-8") as file:
        for event in todays_events:
            file.write(json.dumps(event) + "\n")            # add to the end; never rewrite
```

Use the sample day: F001 bread is picked up for Hope Children's Centre, F002 rice is delivered, F003 bananas were rejected ("no cold storage for fruit today"), and F004 curry is approved and waiting for the Old Market Elders' Group. The day has 16 events.

1. Which offer or person is harmed by Version A? Point at the line.
2. If the power cuts halfway through the save, what is left with each version? Run the experiment below.
3. How would a test catch each problem?

🔌 **Pull the plug.** Yesterday's two offers are saved in two files. Today's save of three offers stops after 43 characters, in the middle of the second offer. Run the cell and read the result.

In [ ]:
import json

yesterday = [{"id": "F001", "portions": 30}, {"id": "F002", "portions": 18}]
for path in ["risky.json", "safe.json"]:
    with open(path, "w", encoding="utf-8") as file:
        json.dump(yesterday, file)

half_text = '[{"id": "F001", "portions": 30}, {"id": "F0'      # today's save, cut off by the power cut

with open("risky.json", "w", encoding="utf-8") as file:      # Version A writes straight into the file
    file.write(half_text)
with open("safe.json.tmp", "w", encoding="utf-8") as file:   # Version B writes the temporary file,
    file.write(half_text)                                    # and os.replace never runs

for path in ["risky.json", "safe.json"]:
    try:
        with open(path, encoding="utf-8") as file:
            print(path, "→", json.load(file))
    except json.JSONDecodeError as error:   # the error from Session 1; "as error" names it, so you can print its message
        print(path, "→ broken:", error)

📝 **Problems in Version A** (which line, and which offer or person is harmed): 1. … · 2. …

📝 **After a power cut halfway** (what is left with each version): …

📝 **How a test catches each problem:** …

<details><summary>Instructor's view</summary>

1. **It writes straight over the only copy.** `open("offers.json", "w")` empties the file the moment it opens. If the power cuts halfway, the file cannot be loaded at all (the experiment shows it). All four offers are lost, not only today's change: F004's curry, which the Old Market Elders' Group is still waiting for, and F001's bread for Hope Children's Centre. Version B writes a complete temporary file first; if it stops, the old `offers.json` is still whole.
2. **It rewrites the history and throws earlier days away.** `open("history.jsonl", "w")` keeps only today's events. Tomorrow evening, today's 16 events are gone, including the reason F003 was rejected ("no cold storage for fruit today"). When Green Market Stall asks next week, nobody can answer. And a history that code may rewrite cannot be trusted: the same code could change an old line, and nobody could show what really happened. A history must be append-only: Version B opens it with `"a"`.
3. Small point: no `encoding="utf-8"`. Khmer text written without it may break on some computers.

**After a power cut halfway:** `risky.json` is broken and cannot be loaded, so yesterday's offers are lost too. `safe.json` still loads yesterday's two offers; only today's change is lost (it was in the temporary file).

**How a test catches it:** save twice and check that the first event is still in the history (the kit's `append_event` check does exactly this); stop a save halfway, as in the experiment, and check that the old file still loads.

**Is Version B perfect?** No: called twice with the same `todays_events`, it writes the events twice. AI code often "simplifies" by dropping exactly the lines that protect data. Ask: *what happens on a bad day?*

</details>

## Session 3 · Build and decide: `storage.py` (90 min)

*New runtime since Session 2? Run the setup cell at the top first.*

### 📁 Your folder now

In [ ]:
bridge.show()

### Write your file

1. Open the **Files** panel (the folder icon on the left) → right-click `my_foodbridge` → **New file** → name it `storage.py`. Double-click `storage.py` to open it, paste the starter below, and save (**Ctrl+S** or **Cmd+S**).
2. **First read `save_records`** (it is complete) and answer the three 🤔 questions under the starter, in the 📝 cell after them.
3. Then fill in TODO 1–3 and save again.

`save_records` is the safe save you read in Session 2. You have practised every other piece. `load_records` is your `load_list` (Try it 3). `append_event` is the first half of your `log_sale` (Try it 4). `read_events` is the puzzle's function plus one line that skips blank lines (`if line.strip() != "":`). The **system** reads `LOG_EVENTS` before it calls `append_event`, so `append_event` writes every event it is given.

```python
# storage.py · Week 7: never lose data, and keep a history
import json
import os

LOG_EVENTS = ["added", "input_rejected", "moved"]   # the kinds of event the system writes to the history


def save_records(records, path):
    """Save every record as JSON. Write a temporary file first, then swap it in."""
    temp_path = path + ".tmp"
    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(records, file, indent=2)
    os.replace(temp_path, path)


def load_records(path):
    """Return the saved records, or [] when nothing is saved yet."""
    # TODO 1: if there is no file at path, return []
    #         otherwise open the file (encoding="utf-8") and return json.load(file)
    return []


def append_event(path, event):
    """Add one event to the end of the history file (one JSON line). Never rewrite earlier lines."""
    # TODO 2: open path with "a" (encoding="utf-8") and write json.dumps(event) + "\n"
    return None


def read_events(path):
    """Return every event in the history file, oldest first, or [] when there is no file."""
    events = []
    # TODO 3: if there is no file at path, return events
    #         otherwise read line by line: skip blank lines (if line.strip() != "":), append json.loads(line)
    return events
```

🤔 **Questions about `save_records`.** Answer in your own words, one or two sentences each:

1. Why does `save_records` write to `path + ".tmp"` first, and not straight to `path`?
2. `save_records` opens a file with `"w"`, but `append_event` must use `"a"`. Why is `"w"` safe here?
3. If the power goes off in the middle of `json.dump`, what is in `offers.json` the next morning?

📝 **My answers:** 1 … · 2 … · 3 …

### ✅ Check each function

Each check works in a temporary folder of its own, so it never touches your real files. `save_records` should pass at once: it is given complete.

In [ ]:
bridge.check("save_records", folder="my_foodbridge")
bridge.check("load_records", folder="my_foodbridge")
bridge.check("append_event", folder="my_foodbridge")
bridge.check("read_events", folder="my_foodbridge")

### 👀 See it

`bridge.show("history")` runs the sample day with **your** `storage.py`, saving into `my_foodbridge/demo_day/`: `offers.json` (replaced after every change) and `history.jsonl` (one line per event). Then a brand-new system loads both files and compares: the **restart test**. An unfinished part is replaced by the model version.

In [ ]:
bridge.show("history")

Each line of `history.jsonl` records the step, the kind of event, the offer, the detail and the time. A real system would also record **who** made the change. `my_foodbridge/demo_day/` is in your Google Drive: open it in the **Files** panel and double-click `history.jsonl`.

### 🧭 Your decision · what does the history keep?

Every event in the history is a promise: *"we can explain this later."* That is **accountability**. But every event also has a cost: a longer history is harder to read, and the one event that matters is easier to miss (**noise**).

| Event | When it happens | What it records | If you drop it |
|---|---|---|---|
| `"added"` | an offer passes the door check | ID, portions, food | nobody can say when an offer arrived |
| `"input_rejected"` | a row is refused at the door | the reason | a donor whose row disappeared gets no answer; the same mistake can repeat unseen |
| `"moved"` | a status changes | old → new status, partner, note | nobody can say who got the food, or why F003 was rejected |

**Your evidence.** The prepared cell below reads `my_foodbridge/demo_day/history.jsonl`, the history that `bridge.show("history")` just saved, and counts each kind of event with the same pattern as your `count_events`. You only read its result.

In [ ]:
# @title Count the events in the saved demo day { display-mode: "form" }
# Reads my_foodbridge/demo_day/history.jsonl and counts each kind of event, like count_events.
import json
import os

history_path = "my_foodbridge/demo_day/history.jsonl"
demo_events = []
if os.path.exists(history_path):
    with open(history_path, encoding="utf-8") as file:
        for line in file:
            if line.strip() != "":
                demo_events.append(json.loads(line))
event_counts = {}
for event in demo_events:
    event_counts[event["event"]] = event_counts.get(event["event"], 0) + 1
print(history_path, "holds", len(demo_events), "events:")
for kind in ["added", "input_rejected", "moved"]:
    print("  ", kind, "→", event_counts.get(kind, "not logged"))

1. Read the counts. Which kind of event is the biggest part of the history? Which kind answers a donor's question?
2. Open `my_foodbridge/storage.py` in the **Files** panel, change `LOG_EVENTS` at the top of the file, and save (**Ctrl+S** or **Cmd+S**). Keep at least one kind, or keep all three if you can explain why. *(With `LOG_EVENTS = []`, no `history.jsonl` is written at all.)*
3. Run the cell below, then the evidence cell again. The check still passes. If you removed a kind, the timeline and the counts change: without `"input_rejected"`, the 5 refusals vanish, and the count says **not logged**.
4. 🤔 How many events are in the history now? Which question can the coordinator no longer answer?

In [ ]:
bridge.check("append_event", folder="my_foodbridge")
bridge.show("history")

📝 **My answer:** …

**Decision log** — *My choice:* `LOG_EVENTS = …` · *An alternative:* … · *Why I chose it:* … · *Evidence (one number, and where it came from):* … (e.g. *5 of 16 events are refusals: the count of my demo_day/history.jsonl, W7*) · *Who is affected / the tradeoff:* accountability ↔ noise: …

### 📓 Project journal

In Week 10 you hand in a short report (`README.md`) about your own system. These journal lines become that report, so write them now, while the week is fresh. Three lines, about five minutes.

📝 *What my system does now:* … · *One decision I made (and why):* … · *One idea for my Final:* …

### ⭐⭐ Go further · a summary of the day's moves

*Optional.* At the end of a week the coordinator wants a short summary, not a long timeline: how many offers were approved, rejected, picked up and delivered? Write `moves_summary(events)`: for every `"moved"` event, find the new status in its `"detail"` (the word after `->`) and count it. Return `{new status: how many}`. Hint: `"-> delivered" in "picked_up -> delivered"` is `True`, because `in` also asks whether a text contains a shorter text. A summary like this is one of the extensions you can choose for your Final. Run the evidence cell above first: it loads `demo_events`, the events of the saved demo day.

In [ ]:
STATUSES = ["approved", "rejected", "picked_up", "delivered", "cancelled"]


def moves_summary(events):
    summary = {}
    # TODO: for each "moved" event, and for each status in STATUSES:
    #       if "-> " + status is in event["detail"], add 1 to summary[status]
    return summary


bridge.check("moves_summary", moves_summary, examples=[
    ([{"event": "moved", "detail": "new -> approved for Hope Children's Centre"},
      {"event": "added", "detail": "30 portions of Bread rolls"},
      {"event": "moved", "detail": "approved -> picked_up"}], {"approved": 1, "picked_up": 1}),
    ([], {}),
])
# demo_events: the saved demo day, loaded by the evidence cell above
print("The saved demo day:", moves_summary(demo_events))

## Finish

- [ ] I can explain memory and disk, and why `open(path, "w")` is risky for a history.
- [ ] I can explain what the temporary file in `save_records` protects, and what a power cut can still lose.
- [ ] My `storage.py` passes all four checks, and `bridge.show("history")` passes the restart test.
- [ ] My decision log explains `LOG_EVENTS` with one number from my saved history, and my journal has three lines.

Your system is always available as a web app: `bridge.start_web(folder="my_foodbridge")`. Its **History** tab is your `history.jsonl`. It reads your files when it starts: after a change, run it again, then reload the page.

📚 **Read more:** [Python Basics](Python_Basics.ipynb) §11 Files and CSV and §12 JSON and JSON Lines, plus one chapter: "Files" in [Python for Everybody (PY4E)](https://www.py4e.com/book) or "Reading and Writing Files" in [Automate the Boring Stuff with Python](https://automatetheboringstuff.com/).

### Research 6 · Code written by AI (5%)

*Share it in the Week 8 class warm-up.* **The concept:** code written by an AI. You ask an AI assistant to write and explain a small function, then you test it with your own cases. About one page.

1. **Read and organise.** Read [Python Basics](Python_Basics.ipynb) §10 Tests with `assert` and §07 Functions and `return`, plus the **"Debugging"** sections of [Think Python (3rd edition)](https://greenteapress.com/wp/think-python-3rd-edition/), and try [Python Tutor](https://pythontutor.com/python-compiler.html) on a short function. Write a summary in your own words: 5–8 sentences on testing and debugging (what a test and a trace are, why they exist, how they work, and one common mistake), and one tiny example you ran, with its output.
2. **Ask an AI.** Ask an AI assistant to write and explain a small function, with a contract as in Week 4 (name, inputs, output, odd cases, one example). Ask at least two follow-up questions: one asks for an everyday example from Cambodia, one asks *"what can go wrong?"*. Keep 6–10 key lines of the conversation.
3. **Compare and check.** Test the AI's function with your own cases (normal, boundary, odd) in Colab, and trace it in Python Tutor. Where was the AI right, vague or wrong?
4. **Your view.** 150–250 words on *"Who is responsible when AI-written code harms someone?"* Take a side and give a reason from this week's case, for example an AI's save function that opened the history with `"w"`.
5. **Sources and honesty.** List the sections and chapters you read, and the AI tool you used.

Use invented examples only, never real personal data. Submit by email to submit.progmis@gmail.com; the deadline and file name are announced in class.

**Next week:** the same records, shown to different people: who may see which fields, the most urgent food first, and honest numbers. You add `my_foodbridge/report.py`, take "the top 3" and "the newest 5" with slices, and draw a chart of your own.

⬅ **Previous:** [Week 6 · Lifecycles](Week_06.ipynb) · **Next:** [Week 8 · Reports, screens and a first chart](Week_08.ipynb) ➡